In [31]:
import json, time, os
import requests
import pandas as pd

from enum import Enum
from typing import Any, List, Dict, Optional, Union

SCRYFALL_BASE_URL = "https://api.scryfall.com/"
os.getcwd()

'c:\\Users\\tallo\\Documents\\programming_projects\\mtg-coding'

# Utils

In [19]:
def write_file(fpath: str, content: Union[str, bytes]) -> None:
    if isinstance(content, str):
        with open(fpath, "wb") as f:
            f.write(content.encode("utf-8"))
    elif isinstance(content, bytes):
        with open(fpath, "wb") as f:
            f.write(content)
    
def read_file(fpath: str, encoding: str = "utf-8") -> str:
    with open(fpath, "r", encoding=encoding) as f:
        return f.read()

# Caching and Requesting

In [ ]:
class CacheHandler:
    def __init__(self, cache_dir: str):
        self.cache_dir = cache_dir

    def _get_key_fpath(self, key: str) -> str:
        return f'{self.cache_dir}/{key}.json'

    def get(self, key: str) -> Optional[str]:
        fpath = self._get_key_fpath(key)
        return read_file(fpath) if os.path.exists(fpath) else None

    def set(self, key: str, value: Union[str, bytes]) -> None:
        fpath = self._get_key_fpath(key)
        write_file(fpath, value)

    def delete(self, key: str) -> None:
        fpath = self._get_key_fpath(key)
        if not os.path.exists(fpath):
            return
        os.remove(fpath)

    def clear(self) -> None:
        files = [os.path.join(self.cache_dir, f) for f in os.listdir(self.cache_dir)]
        [self.delete(f) for f in files]

In [22]:
class ScryfallHttpClient:
    def __init__(self):
        pass

    def _get_scryfall_headers(self) -> Dict[str, str]:
        return {
            "User-Agent": "MTG-DataScience-Project/1.0",
            "Accept": "application/json;q=0.9,*/*,q=0.8",
        }
    
    def get(self, url: str, **kwargs) -> requests.Response:
        response = requests.get(url, headers=self._get_scryfall_headers(), **kwargs)
        if response.status_code != 200:
            response.raise_for_status()
        return response

In [30]:
class ScryfallClient:
    def __init__(self, base_url: str, cache_handler: CacheHandler, http_client: ScryfallHttpClient):
        self.cache_handler = cache_handler
        self.http_client = http_client
        self.base_url = base_url
        self.endpoints = {
            "bulk-data": "bulk-data",
        }

    def get_bulk_data(self) -> pd.DataFrame:
        cache_contents = self.cache_handler.get(self.endpoints['bulk-data'])
        if cache_contents is not None:
            return pd.DataFrame(json.loads(cache_contents).get("data"))

        response_json = self.http_client.get(f"{self.base_url}/{self.endpoints['bulk-data']}")
        self.cache_handler.set(self.endpoints['bulk-data'], json.dumps(response_json.json(), indent=4))
        cache_contents = self.cache_handler.get(self.endpoints['bulk-data'])
        return pd.DataFrame(json.loads(cache_contents).get("data"))

    def get_oracle_cards(self) -> pd.DataFrame:
        cache_key = 'oracle-cards'
        res_df = self.get_bulk_data()
        download_uri = res_df[res_df['name'] == 'Oracle Cards']['download_uri'].iloc[0]

        cache_contents = self.cache_handler.get(cache_key)
        if cache_contents is not None:
            return pd.DataFrame(json.loads(cache_contents))

        response = self.http_client.get(download_uri, stream=True)
        self.cache_handler.set(cache_key, response.content)
        cache_contents = self.cache_handler.get(cache_key) 
        return pd.DataFrame(json.loads(cache_contents))


cache_handler = CacheHandler("./cache/")
scryfall_http_client = ScryfallHttpClient()
sfc = ScryfallClient(SCRYFALL_BASE_URL, cache_handler, scryfall_http_client)
# sfc.get_bulk_data()
sfc.get_oracle_cards().head(1).columns
# print(json.dumps(json.loads(cache_handler.get('oracle-cards').split("\n")[1][:-1]), indent=4))

Index(['object', 'id', 'oracle_id', 'multiverse_ids', 'mtgo_id',
       'tcgplayer_id', 'cardmarket_id', 'name', 'lang', 'released_at', 'uri',
       'scryfall_uri', 'layout', 'highres_image', 'image_status', 'image_uris',
       'mana_cost', 'cmc', 'type_line', 'oracle_text', 'power', 'toughness',
       'colors', 'color_identity', 'keywords', 'all_parts', 'legalities',
       'games', 'reserved', 'game_changer', 'foil', 'nonfoil', 'finishes',
       'oversized', 'promo', 'reprint', 'variation', 'set_id', 'set',
       'set_name', 'set_type', 'set_uri', 'set_search_uri', 'scryfall_set_uri',
       'rulings_uri', 'prints_search_uri', 'collector_number', 'digital',
       'rarity', 'watermark', 'flavor_text', 'card_back_id', 'artist',
       'artist_ids', 'illustration_id', 'border_color', 'frame',
       'frame_effects', 'security_stamp', 'full_art', 'textless', 'booster',
       'story_spotlight', 'edhrec_rank', 'preview', 'prices', 'related_uris',
       'purchase_uris', 'mtgo_foil_i

In [ ]:
first_lines = file_contents.split("\n")[1][:-1]
print(json.dumps(json.loads(first_lines), indent=4))